In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
df=pd.read_csv('HR_data_2.csv',index_col=0)

In [4]:
features=['HR_TD_Mean','HR_TD_std','TEMP_TD_Mean','TEMP_TD_std','EDA_TD_P_Mean','EDA_TD_T_Mean']
df_clean=df.dropna(subset=features+['Phase', 'Frustrated']).copy()
print("clean")

clean


In [5]:
train=df_clean[df_clean['Phase']=='phase1'][features]
scaler=StandardScaler()
X_train=scaler.fit_transform(train)
model=OneClassSVM(kernel='rbf')
model.fit(X_train)
print("trained")

trained


vi skal lige vælge rigtig kernal og måske andet, RBF bliver brugt i slides


In [6]:
p2=df_clean[df_clean['Phase']=='phase2'].copy()
p2['pred']=model.predict(scaler.transform(p2[features]))
flagged=(p2['pred']==-1).sum()
print(f"phase2 anomaly %:{flagged}/{len(p2)} ({100*flagged/len(p2):.1f}%)")

phase2 anomaly %:45/104 (43.3%)


In [7]:
subjects=df_clean['Individual'].unique()
results=[]
for subj in subjects:
    train=df_clean[(df_clean['Individual']!=subj)&(df_clean['Phase']=='phase1')][features]
    test=df_clean[(df_clean['Individual']==subj)&(df_clean['Phase']=='phase2')][features]
    if len(train)<5 or len(test)==0:
        continue
    sc=StandardScaler().fit(train)
    m=OneClassSVM(kernel='rbf').fit(sc.transform(train))
    preds=m.predict(sc.transform(test))
    flagged=(preds==-1).sum()
    results.append({'subject':subj,'n_test':len(test),'flagged':flagged,'pct':100*flagged/len(test)})
loso=pd.DataFrame(results)
print(loso.to_string(index=False))
print(f"\nmean anomaly rate: {loso['pct'].mean():.1f}%")

 subject  n_test  flagged   pct
       1       4        1  25.0
       2       4        0   0.0
       3       4        1  25.0
       4       4        1  25.0
       5       4        1  25.0
       6       4        4 100.0
       7       4        0   0.0
       8       4        4 100.0
       9       4        2  50.0
      10       4        3  75.0
      11       4        2  50.0
      12       4        0   0.0
      13       4        0   0.0
      14       4        0   0.0
      15       4        4 100.0
      16       4        4 100.0
      17       4        4 100.0
      18       4        4 100.0
      19       4        4 100.0
      20       4        4 100.0
      21       4        0   0.0
      22       4        1  25.0
      23       4        2  50.0
      24       4        1  25.0
      25       4        1  25.0
      26       4        3  75.0

mean anomaly rate: 49.0%
